# LSTM
- 기억셀을 도입하여 단기기억을 길게 유지할 수 있음
    - 기억셀-기억셀의 역전파경로에서는 이전 $f$의 아다마르 곱으로 구성
    - 이건 2회차에 수식 이나 데이터로 어느정도까지 되나 보자고
- 게이트 값에 시그모이드/아다마르 곱을 써서 마스크로 동작
- f, i, o, g, c, h를 통해 계산
    - $f$: forget gate 
        - cell의 정보에 대한 유지율
        - $c_{t-1}$에 대한 마스크
        - $$f = \sigma(x_tW_x^{(f)}+h_{t-1}W_h^{(f)}+b^{(f)})$$
    - $i$: input gate
        - x, h에 의한 활성 중 기억셀에 반영할 내용 선택
        - $g$에 대한 마스크
        - $$i = \sigma(x_tW_x^{(i)}+h_{t-1}W_h^{(i)}+b^{(i)})$$
    - $o$: output gate
        - cell에 의한 활성 중 이번 시점의 hidden이 될 내용 결정
        - $\tanh(c_{t-1})$에 대한 마스크
        - $$o = \sigma(x_tW_x^{(o)}+h_{t-1}W_h^{(o)}+b^{(o)})$$
    - $g$: cell input activation
        - cell에 추가할 정보 후보
        - 직전 상태와 입력에 의한 활성
        - $$g = \tanh(x_tW_x^{(g)}+h_{t-1}W_h^{(g)}+b^{(g)})$$
    - $c$: cell memory
        - 이전 상태들을 누적 저장해온 내부 메모리
        - 이전 내부 메모리에서 $f$를 통해 일부 정보를 지우고, $g$, $i$를 통해 정보를 추가 
        - $$c_t = f\odot c_{t-1}+g\odot i$$
    - $h$: hidden state
        - 현재 시점에 외부로 드러나는 상태
        - $c_t$의 $\tanh$활성 중 o를 통해 노출할 정보를 결정
        - $$h_t = o\odot tanh(c_t)$$


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import cupy as np 
import matplotlib.pyplot as plt

## 구현
- 각 가중치들 concat으로 연결 - 한번에 모든 게이트 및 활성에 대한 아핀 변환 계산 가능
    - $$W_x \in \mathbb R ^{D\times 4H}$$
    - $$W_h \in \mathbb R ^{H\times 4H}$$
    - $$b \in \mathbb R ^{1 \times 4H}$$

In [ ]:
from b2.common.functions import sigmoid


class LSTM:
    def __init__(self, Wx, Wh, b):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.cache = None

    def forward(self, x, h_prev, c_prev):
        Wx, Wh, b = self.params
        N, H = h_prev.shape

        A = x @ Wx + h_prev @ Wh + b
        f = A[:, :H]
        g = A[:, H : 2 * H]
        i = A[:, 2 * H : 3 * H]
        o = A[:, 3 * H :]

        f = sigmoid(f)
        g = np.tanh(g)
        i = sigmoid(i)
        o = sigmoid(o)

        c_next = f * c_prev + g * i
        h_next = o * np.tanh(c_next)

        self.cache = (x, h_prev, c_prev, i, f, g, o, c_next)
        return h_next, c_next

    def backward(self, dh_next, dc_next):
        Wx, Wh, b = self.params
        x, h_prev, c_prev, i, f, g, o, c_next = self.cache

        tanh_c_next = np.tanh(c_next)

        ds = dc_next + (dh_next * o) * (1 - tanh_c_next**2)

        dc_prev = ds * f

        di = ds * g
        df = ds * c_prev
        do = dh_next * tanh_c_next
        dg = ds * i

        di *= i * (1 - i)
        df *= f * (1 - f)
        do *= o * (1 - o)
        dg *= 1 - g**2

        dA = np.hstack((df, dg, di, do))

        dWh = np.dot(h_prev.T, dA)
        dWx = np.dot(x.T, dA)
        db = dA.sum(axis=0)

        self.grads[0][...] = dWx
        self.grads[1][...] = dWh
        self.grads[2][...] = db

        dx = np.dot(dA, Wx.T)
        dh_prev = np.dot(dA, Wh.T)

        return dx, dh_prev, dc_prev

In [ ]:
class TimeLSTM:
    def __init__(self, Wx, Wh, b, stateful=False):
        self.params = [Wx, Wh, b]
        self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
        self.layers = None
        self.cache = None
        self.h, self.c = None, None
        self.dh = None
        self.stateful = stateful

    def forward(self, xs):
        Wx, Wh, b = self.params
        N, T, D = xs.shape
        H = Wh.shape[0]

        self.layers = []
        hs = np.empty((N, T, H), dtype="f")

        if not self.stateful or self.h is None:
            self.h = np.zeros((N, H), dtype="f")
        if not self.stateful or self.c is None:
            self.c = np.zeros((N, H), dtype="f")

        for t in range(T):
            layer = LSTM(*self.params)
            self.h, self.c = layer.forward(xs[:, t, :], self.h, self.c)
            hs[:, t, :] = self.h

            self.layers.append(layer)

        return hs

    def backward(self, dhs):
        Wx, Wh, b = self.params
        N, T, H = dhs.shape
        D = Wx.shape[0]

        dxs = np.empty((N, T, D), dtype="f")
        dh, dc = 0, 0

        grads = [0, 0, 0]
        for t in reversed(range(T)):
            layer = self.layers[t]
            dx, dh, dc = layer.backward(dhs[:, t, :] + dh, dc)
            dxs[:, t, :] = dx
            for i, grad in enumerate(layer.grads):
                grads[i] += grad

        for i, grad in enumerate(grads):
            self.grads[i][...] = grad
            self.dh = dh
        return dxs

    def set_state(self, h, c=None):
        self.h, self.c = h, c

    def reset_state(self):
        self.h, self.c = None, None

In [ ]:
import pickle
from b2.common.time_layers import TimeEmbedding, TimeAffine, TimeSoftmaxWithLoss


class Rnnlm:
    def __init__(self, vocab_size=10000, wordvec_size=100, hidden_size=100):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype("f")
        lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype("f")
        lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype("f")
        lstm_b = np.zeros(4 * H).astype("f")
        affine_W = (rn(H, V) / np.sqrt(H)).astype("f")
        affine_b = np.zeros(V).astype("f")

        self.layers = [
            TimeEmbedding(embed_W),
            TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=True),
            TimeAffine(affine_W, affine_b),
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.lstm_layer = self.layers[1]

        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs):
        for layer in self.layers:
            xs = layer.forward(xs)
        return xs

    def forward(self, xs, ts):
        score = self.predict(xs)
        loss = self.loss_layer.forward(score, ts)
        return loss

    def backward(self, dout=1):
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        self.lstm_layer.reset_state()

    def save_params(self, file_name="Rnnlm.pkl"):
        with open(file_name, "wb") as f:
            pickle.dump(self.params, f)

    def load_params(self, file_name="Rnnlm.pkl"):
        with open(file_name, "rb") as f:
            self.param = pickle.load(f)


In [ ]:
from datasets import ptb
from b2.common.optimizer import SGD
from b2.common.trainer import RnnlmTrainer
from b2.common.util import eval_perplexity

batch_size = 20
wordvec_size = 100
hidden_size = 100
time_size = 35
lr = 20.0
max_epoch = 4
max_grad = 0.25


corpus, word_to_id, id_to_word = ptb.load_data("train")
corpus_test, _, _ = ptb.load_data("test")
vocab_size = len(word_to_id)
xs = corpus[:-1]
ts = corpus[1:]

model = Rnnlm(vocab_size, wordvec_size, hidden_size)
optimizer = SGD(lr)
trainer = RnnlmTrainer(model, optimizer)

# 기울기 클리핑을 적용하여 학습
trainer.fit(xs, ts, max_epoch, batch_size, time_size, max_grad, eval_interval=20)
trainer.plot(ylim=(0, 500))

# 테스트 데이터로 평가
model.reset_state()
ppl_test = eval_perplexity(model, corpus_test)
print("테스트 퍼플렉서티: ", ppl_test)

# 매개변수 저장
model.save_params()


## 다층화
- 여러층의 LSTM 적층
- 과대적합 가능성 높음
    - 정규화 적용
    - 변형 드롭아웃
        - 시간/깊이 방향 양쪽으로 적용
        - 레이어별 동일 마스크 적용 - 정보의 지수적 손실 방지
        - 일반적인 드롭아웃을 시간방향으로 적용시, 드롭아웃에 의해 시간방향 노이즈 누적으로 정보 손실
    - 가중치 공유
        - 서로 다른 레이어의 가중치 공유
        - 학습 파라미터 감소 - 과대적합 억제
        - embedding과 affine의 가중치 공유
            - embedding 계층 - 단어 벡터를 꺼냄
            - affine 계층 - 문맥 벡터에서 단어별 점수를 꺼냄
            - embedding 계층 가중치가 단어 벡터 stack이므로, affine 계층의 가중치로 사용시 hidden과 단어 벡터별 내적 stack으로 변환
        - 

In [ ]:
from b2.common.time_layers import TimeDropout

class BetterRnnlm():
    def __init__(self, vocab_size=10000, wordvec_size=650,
                 hidden_size=650, dropout_ratio=0.5):
        V, D, H = vocab_size, wordvec_size, hidden_size
        rn = np.random.randn

        embed_W = (rn(V, D) / 100).astype('f')
        lstm_Wx1 = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
        lstm_Wh1 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b1 = np.zeros(4 * H).astype('f')
        lstm_Wx2 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_Wh2 = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
        lstm_b2 = np.zeros(4 * H).astype('f')
        affine_b = np.zeros(V).astype('f')

        self.layers = [
            TimeEmbedding(embed_W),
            TimeDropout(dropout_ratio),
            TimeLSTM(lstm_Wx1, lstm_Wh1, lstm_b1, stateful=True),
            TimeDropout(dropout_ratio),
            TimeLSTM(lstm_Wx2, lstm_Wh2, lstm_b2, stateful=True),
            TimeDropout(dropout_ratio),
            TimeAffine(embed_W.T, affine_b)  # weight tying!!
        ]
        self.loss_layer = TimeSoftmaxWithLoss()
        self.lstm_layers = [self.layers[2], self.layers[4]]
        self.drop_layers = [self.layers[1], self.layers[3], self.layers[5]]

        self.params, self.grads = [], []
        for layer in self.layers:
            self.params += layer.params
            self.grads += layer.grads

    def predict(self, xs, train_flg=False):
        for layer in self.drop_layers:
            layer.train_flg = train_flg

        for layer in self.layers:
            xs = layer.forward(xs)
        return xs

    def forward(self, xs, ts, train_flg=True):
        score = self.predict(xs, train_flg)
        loss = self.loss_layer.forward(score, ts)
        return loss

    def backward(self, dout=1):
        dout = self.loss_layer.backward(dout)
        for layer in reversed(self.layers):
            dout = layer.backward(dout)
        return dout

    def reset_state(self):
        for layer in self.lstm_layers:
            layer.reset_state()

    def save_params(self, file_name="bRnnlm.pkl"):
        with open(file_name, "wb") as f:
            pickle.dump(self.params, f)

    def load_params(self, file_name="bRnnlm.pkl"):
        with open(file_name, "rb") as f:
            self.param = pickle.load(f)

In [26]:
from b2.common import config
from b2.common.util import to_gpu

# 하이퍼파라미터 설정
batch_size = 80
wordvec_size = 650
hidden_size = 650
time_size = 35
lr = 20.0
max_epoch = 40
max_grad = 0.25
dropout = 0.5

# 학습 데이터 읽기
corpus, word_to_id, id_to_word = ptb.load_data('train')
corpus_val, _, _ = ptb.load_data('val')
corpus_test, _, _ = ptb.load_data('test')

if config.GPU:
    corpus = to_gpu(corpus)
    corpus_val = to_gpu(corpus_val)
    corpus_test = to_gpu(corpus_test)

vocab_size = len(word_to_id)
xs = corpus[:-1]
ts = corpus[1:]

model = BetterRnnlm(vocab_size, wordvec_size, hidden_size, dropout)
optimizer = SGD(lr)
trainer = RnnlmTrainer(model, optimizer)

best_ppl = float('inf')
for epoch in range(max_epoch):
    trainer.fit(xs, ts, max_epoch=1, batch_size=batch_size,
                time_size=time_size, max_grad=max_grad)

    model.reset_state()
    ppl = eval_perplexity(model, corpus_val)
    print('검증 퍼플렉서티: ', ppl)

    if best_ppl > ppl:
        best_ppl = ppl
        model.save_params()
    else:
        lr /= 4.0
        optimizer.lr = lr

    model.reset_state()
    print('-' * 50)


# 테스트 데이터로 평가
model.reset_state()
ppl_test = eval_perplexity(model, corpus_test)
print('테스트 퍼플렉서티: ', ppl_test)


| 에폭 35 |  반복 181 / 331 | 시간 105[s] | 퍼플렉서티 50.51
| 에폭 35 |  반복 201 / 331 | 시간 116[s] | 퍼플렉서티 47.15
| 에폭 35 |  반복 221 / 331 | 시간 128[s] | 퍼플렉서티 44.24
| 에폭 35 |  반복 241 / 331 | 시간 139[s] | 퍼플렉서티 45.51
| 에폭 35 |  반복 261 / 331 | 시간 150[s] | 퍼플렉서티 47.89
| 에폭 35 |  반복 281 / 331 | 시간 162[s] | 퍼플렉서티 48.91
| 에폭 35 |  반복 301 / 331 | 시간 173[s] | 퍼플렉서티 47.56
| 에폭 35 |  반복 321 / 331 | 시간 184[s] | 퍼플렉서티 43.34
퍼플렉서티 평가 중 ...
209 / 210
검증 퍼플렉서티:  84.04309
--------------------------------------------------
| 에폭 36 |  반복 1 / 331 | 시간 0[s] | 퍼플렉서티 62.47
| 에폭 36 |  반복 21 / 331 | 시간 13[s] | 퍼플렉서티 52.00
| 에폭 36 |  반복 41 / 331 | 시간 25[s] | 퍼플렉서티 49.40
| 에폭 36 |  반복 61 / 331 | 시간 38[s] | 퍼플렉서티 46.45
| 에폭 36 |  반복 81 / 331 | 시간 50[s] | 퍼플렉서티 42.03
| 에폭 36 |  반복 101 / 331 | 시간 63[s] | 퍼플렉서티 39.26
| 에폭 36 |  반복 121 / 331 | 시간 75[s] | 퍼플렉서티 44.38
| 에폭 36 |  반복 141 / 331 | 시간 88[s] | 퍼플렉서티 47.74
| 에폭 36 |  반복 161 / 331 | 시간 98[s] | 퍼플렉서티 46.33
| 에폭 36 |  반복 181 / 331 | 시간 109[s] | 퍼플렉서티 50.47
| 에폭 36 |  반복 201 / 

In [ ]:
from b2.common.functions import softmax
from b2.common.models import Rnnlm


class RnnlmGen(Rnnlm):
    def generate(self, start_id, skip_ids=None, sample_size=100):
        word_ids = [start_id]

        x = start_id
        while len(word_ids) < sample_size:
            x = np.array(x).reshape(1, 1)
            score = self.predict(x)
            p = softmax(score.flatten())

            sampled = np.random.choice(len(p), size=1, p=p)
            if (skip_ids is None) or (sampled not in skip_ids):
                x = sampled
                word_ids.append(int(x[0]))

        return word_ids

    def get_state(self):
        return self.lstm_layer.h, self.lstm_layer.c

    def set_state(self, state):
        self.lstm_layer.set_state(*state)


In [ ]:
from datasets import ptb


corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)
corpus_size = len(corpus)

model = RnnlmGen()
model.load_params('Rnnlm.pkl')

start_word = 'you'
start_id = word_to_id[start_word]
skip_words = ['N', '<unk>', '$']
skip_ids = [word_to_id[w] for w in skip_words]
word_ids = model.generate(start_id, skip_ids)
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')
print(txt)


you can take once to all the outsider 's problems baldwin and has resignation by in papers.
 mr. jones is n't brought to the buildup of how mr. lee who will get mr. roman 's messages to honor his partner.
 he is one of his senators out of the concerns and he would n't match the staggering final condition of running changes.
 he is also controversial will gain posts at walter phillips 's takeover expert.
 worked at the accord which scrambled to revive automated second northeast 's chairman said.
 mr. roman says he is n't


In [ ]:
from b2.common.models import BetterRnnlm


class BetterRnnlmGen(BetterRnnlm):
    def generate(self, start_id, skip_ids=None, sample_size=100):
        word_ids = [start_id]

        x = start_id
        while len(word_ids) < sample_size:
            x = np.array(x).reshape(1, 1)
            score = self.predict(x).flatten()
            p = softmax(score).flatten()

            sampled = np.random.choice(len(p), size=1, p=p)
            if (skip_ids is None) or (sampled not in skip_ids):
                x = sampled
                word_ids.append(int(x[0]))

        return word_ids

    def get_state(self):
        states = []
        for layer in self.lstm_layers:
            states.append((layer.h, layer.c))
        return states

    def set_state(self, states):
        for layer, state in zip(self.lstm_layers, states):
            layer.set_state(*state)


In [ ]:

corpus, word_to_id, id_to_word = ptb.load_data('train')
vocab_size = len(word_to_id)
corpus_size = len(corpus)


model = BetterRnnlmGen()
model.load_params('bRnnlm.pkl')

start_word = 'you'
start_id = word_to_id[start_word]
skip_words = ['N', '<unk>', '$']
skip_ids = [word_to_id[w] for w in skip_words]

word_ids = model.generate(start_id, skip_ids)
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')

print(txt)


model.reset_state()

start_words = 'the meaning of life is'
start_ids = [word_to_id[w] for w in start_words.split(' ')]

for x in start_ids[:-1]:
    x = np.array(x).reshape(1, 1)
    model.predict(x)

word_ids = model.generate(start_ids[-1], skip_ids)
word_ids = start_ids[:-1] + word_ids
txt = ' '.join([id_to_word[i] for i in word_ids])
txt = txt.replace(' <eos>', '.\n')
print('-' * 50)
print(txt)


you 're going to do that says michael hart 's son.
 other aftermath of the town makes well and dr. johnson 's district said the segment 's cost of social firms is structural growth lobbyists do n't realize expanded rates back.
 it may also be the best way to stress that real outlays she says is it thought to be a factor for further injury.
 the survey found that residential fourth times showed that the agency found lower production volume will pick to raise prices.
 and the senate has already picked working on interim tax payments
--------------------------------------------------
the meaning of life is in the group 's position and a big political process.
 but in fact mr. adams offers that a sales base in the u.s. is moving how michigan to pay up the first year.
 house.
 pesetas would go up a smaller and more reasonable financial backers.
 small-business specialists listen to dinkins a fancy young and says they are only dance a harvard here know.
 just we 're in the middle of this tim